In [ ]:
import os
import psycopg

# Uses standard libpq environment variables. Supply credentials outside the notebook.
conn = psycopg.connect(
    dbname=os.getenv("PGDATABASE", "openalex_philosophy"),
    user=os.getenv("PGUSER", "postgres"),
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432")
)

print("Connection successful!")

In [ ]:
%run ./transform.ipynb

In [ ]:
# create the loading function

from psycopg import sql

def load_records(conn, records, table_name, columns):

    # STEP 1: Construct the SQL INSERT statement.
    #
    # sql.SQL() defines the structure of the query.
    # The {} placeholders will be replaced by the table name,
    # column names, and value placeholders.

    query = sql.SQL(
        "INSERT INTO {} ({}) VALUES ({}) ON CONFLICT DO NOTHING"
    ).format(

        # STEP 2: Insert the table name.
        #
        # sql.Identifier() safely represents a PostgreSQL identifier.
        # For example, table_name = "source" becomes "source".

        sql.Identifier(table_name),

        # STEP 3: Insert the column names.
        #
        # map(sql.Identifier, columns) converts each column name
        # into a safe SQL identifier.
        #
        # sql.SQL(", ").join() separates them with commas.
        #
        # Example:
        # ["id", "name"] becomes "id", "name".

        sql.SQL(", ").join(map(sql.Identifier, columns)),

        # STEP 4: Create the value placeholders.
        #
        # sql.Placeholder() represents a %s placeholder.
        #
        # len(columns) determines how many placeholders we need.
        #
        # Example:
        # If columns = ["id", "name"], len(columns) = 2.
        #
        # The resulting SQL fragment is: %s, %s

        sql.SQL(", ").join(
            [sql.Placeholder()] * len(columns)
        )
    )

    # STEP 5: Execute the INSERT statement.
    #
    # try/except allows us to handle database errors.
    # If something goes wrong, we can roll back the transaction.

    try:

        # Open a cursor to communicate with PostgreSQL.
        # The with statement automatically closes the cursor.

        with conn.cursor() as cur:

            # Iterate through every dictionary in records.
            #
            # Example:
            # records = [
            #     {"id": "S001", "name": "Journal A"},
            #     {"id": "S002", "name": "Journal B"}
            # ]

            for record in records:

                # STEP 6: Extract values in the correct column order.
                #
                # record.get(col) retrieves the value corresponding
                # to each column name.
                #
                # Example:
                # columns = ["id", "name"]
                # values = ("S001", "Journal A")
                #
                # Using columns ensures that the order of the values
                # matches the order in the SQL INSERT statement.

                values = tuple(record.get(col) for col in columns)

                # STEP 7: Execute the query.
                #
                # The query contains the SQL structure.
                # The values tuple contains the actual data.
                #
                # psycopg safely substitutes the values into
                # the placeholders.

                cur.execute(query, values)

        # STEP 8: Commit the transaction.
        #
        # conn.commit() saves all successful insertions
        # performed during this function call.

        conn.commit()

    except Exception:

        # STEP 9: Roll back if an error occurs.
        #
        # If any insertion fails, rollback undoes all changes
        # made in the current transaction.
        #
        # This prevents partially completed table loads.

        conn.rollback()

        # Raise the original error so we can investigate it.

        raise

In [ ]:
# Load the tables

# 1. SOURCE
#
# The source table has no foreign-key dependencies.
# Each record contains a source ID and its name.

load_records(
    conn,
    source,
    "source",
    ["id", "name"]
)


# 2. AUTHOR
#
# The author table has no foreign-key dependencies.
# The orcid column is optional and may contain None.

load_records(
    conn,
    author,
    "author",
    ["id", "name", "orcid"]
)


# 3. INSTITUTION
#
# The institution table has no foreign-key dependencies.
# It stores institution IDs and names.

load_records(
    conn,
    institution,
    "institution",
    ["id", "name"]
)


# 4. TOPIC
#
# The topic table has no foreign-key dependencies.
# It stores research topics and their classifications.

load_records(
    conn,
    topic,
    "topic",
    ["id", "name", "field", "subfield", "domain"]
)


# 5. PAPER
#
# The paper table contains a foreign key referencing source.
#
# We load source first to ensure that referenced
# source IDs already exist in PostgreSQL.

load_records(
    conn,
    paper,
    "paper",
    ["id", "title", "year", "is_oa", "type", "source_id"]
)


# 6. AUTHORSHIP
#
# This is a relationship table connecting authors and papers.
#
# Both the author and paper tables must already be populated.

load_records(
    conn,
    authorship,
    "authorship",
    ["author_id", "paper_id"]
)


# 7. PAPER_TOPIC
#
# This relationship table connects papers to research topics.
#
# Both paper and topic must already be populated.

load_records(
    conn,
    paper_topic,
    "paper_topic",
    ["paper_id", "topic_id"]
)


# 8. INST_AUTH
#
# This table connects institutions to specific authorships.
#
# It references institution and the composite key
# (author_id, paper_id) in authorship.
#
# Therefore, it must be loaded after institution
# and authorship.

load_records(
    conn,
    inst_auth,
    "inst_auth",
    ["inst_id", "author_id", "paper_id"]
)

In [ ]:
# verify all 8 tables and count rows

tables = [
    "source", "author", "institution", "topic",
    "paper", "authorship", "paper_topic", "inst_auth"
]

with conn.cursor() as cur:
    for table in tables:
        query = sql.SQL("SELECT COUNT(*) FROM {}").format(
            sql.Identifier(table)
        )
        cur.execute(query)
        print(f"{table}: {cur.fetchone()[0]} rows")